# Silver — quality and canonical keys

Enforce the public schema, exact synthetic dataset hash, finite timings, unique session/participant/lap keys and complete population. No invalid record is silently dropped. In Workbench, read the preceding Delta layer, not a hidden local source.


In [ ]:
from pathlib import Path
import json, os, sys
root_hint = os.environ.get("AIDP_PROJECT_ROOT")
if root_hint:
    root = Path(root_hint).resolve()
else:
    root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "race_demo/pipeline.py").is_file()), None)
if root is None or not (root / "race_demo/pipeline.py").is_file():
    raise RuntimeError("Set AIDP_PROJECT_ROOT to this repository checkout.")
sys.path.insert(0, str(root))
from race_demo.pipeline import generate_bronze, build_silver, build_gold, serving_document, write_once, digest
from race_demo.workbench import publish_table, table_name
execution = os.environ.get("RACE_DEMO_EXECUTION", "LOCAL_DEMO")
if execution not in {"LOCAL_DEMO", "AIDP_WORKBENCH"}:
    raise ValueError("Unknown execution mode")
if execution == "AIDP_WORKBENCH":
    if "spark" not in globals():
        raise RuntimeError("Use an AIDP notebook runtime with Spark/Delta configured.")
    table_name("bronze")  # Validate the dedicated demo namespace before writes.
    if not os.environ.get("RACE_DEMO_OUTPUT"):
        raise RuntimeError("Set RACE_DEMO_OUTPUT to a new writable demo volume directory.")
output = Path(os.environ.get("RACE_DEMO_OUTPUT", str(root / "build/notebooks")))
print("Execution:", execution, "| Evidence: SYNTHETIC_DEMO")


In [ ]:
bronze = ([r.asDict() for r in spark.table(table_name("bronze")).collect()]
          if execution == "AIDP_WORKBENCH" else json.loads((output / "bronze/laps.json").read_text()))
silver = build_silver(bronze)
write_once(output / "silver/laps.json", silver)
if execution == "AIDP_WORKBENCH":
    print(publish_table(spark, "silver", silver))
print({"rows": len(silver), "unique_keys": len({(r["session_id"], r["driver_id"], r["lap"]) for r in silver})})
